# Qwen3.5-2B as a Security Camera Analyst

Can a small, general video VLM watch a security camera clip, describe what happens, and decide on its own whether an alert rule was triggered, staying quiet when nothing happens?

[Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B) is a 2B instruction-following vision-language model (Apache-2.0) with native video input. We test it on a few real CCTV clips:

- **5 event clips** from [UCF-Crime](https://www.crcv.ucf.edu/projects/real-world/): break-in, fighting, shoplifting, vehicle theft and robbery.
- **4 normal clips** from UCF-Crime's normal footage: a mall, a busy supermarket checkout, a hotel lobby and a bowling alley.
- **1 ambiguous clip**: an arrest (a chase and scuffle in a lobby), reported but not scored.

Every clip gets the same prompt with the camera's alert rules. We check the description, the triggered rules, and the time per clip, with thinking off and on.

## Setup

UCF-Crime is released for research use; the clips are downloaded into `data/` (git-ignored) from the [`mteb/ucf-crime`](https://huggingface.co/datasets/mteb/ucf-crime) mirror.

In [ ]:
import json
import re
import shutil
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModelForImageTextToText, AutoProcessor
from transformers.video_utils import VideoMetadata

print("torch", torch.__version__, "| MPS available:", torch.backends.mps.is_available())

## Test clips

Each event clip is cut from 5 s before to 5 s after the annotated event; normal clips are used whole. `expected` is the rule that should fire (`None` for normal clips).

In [ ]:
DATA_DIR = Path("data/raw")

# name, category folder, (start s, end s) or None for the whole video, expected rule
CLIPS = [
    ("Burglary092", "Burglary", (3.0, 19.0), "break_in"),
    ("Fighting033", "Fighting", (14.0, 33.0), "fighting"),
    ("Shoplifting016", "Shoplifting", (16.0, 29.0), "shoplifting"),
    ("Stealing036", "Stealing", (37.0, 58.0), "robbery_or_theft"),
    ("Robbery106", "Robbery", (11.0, 25.0), "robbery_or_theft"),
    ("Normal_Videos_015", "Testing_Normal", None, None),
    ("Normal_Videos_063", "Testing_Normal", None, None),
    ("Normal_Videos_345", "Testing_Normal", None, None),
    ("Normal_Videos_745", "Testing_Normal", None, None),
    ("Arrest001", "Arrest", (34.5, 54.5), "ambiguous"),
]

for name, folder, _, _ in CLIPS:
    target = DATA_DIR / f"{name}_x264.mp4"
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(hf_hub_download("mteb/ucf-crime", f"videos/{folder}/{name}_x264.mp4", repo_type="dataset"), target)


Preprocessing follows the Marlin spike: decode with OpenCV (no FFmpeg needed), sample at 2 fps, and resize so each side is a multiple of 32 within a ~448×448 pixel budget. These UCF clips are only 320×240, so they're passed at native size (320×224).

In [ ]:
SAMPLE_FPS = 2.0
MAX_PIXELS = 200_704
FACTOR = 32


def load_clip(name, window):
    capture = cv2.VideoCapture(str(DATA_DIR / f"{name}_x264.mp4"))
    source_fps = capture.get(cv2.CAP_PROP_FPS)
    total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    start, end = (0, total) if window is None else (int(window[0] * source_fps), min(total, int(window[1] * source_fps)))
    keep = set(np.round(np.arange(start, end, source_fps / SAMPLE_FPS)).astype(int).tolist())

    frames = []
    for index in range(end):
        ok, frame = capture.read()
        if not ok:
            break
        if index in keep:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    height, width = frames[0].shape[:2]
    scale = min(1.0, (MAX_PIXELS / (height * width)) ** 0.5)
    new_h, new_w = max(FACTOR, int(height * scale) // FACTOR * FACTOR), max(FACTOR, int(width * scale) // FACTOR * FACTOR)
    frames = [cv2.resize(f, (new_w, new_h), interpolation=cv2.INTER_AREA) for f in frames]
    if len(frames) % 2:
        frames.append(frames[-1])
    return np.stack(frames)


videos = {name: load_clip(name, window) for name, _, window, _ in CLIPS}

fig, axes = plt.subplots(len(CLIPS), 5, figsize=(14, 2.1 * len(CLIPS)))
for row, (name, _, _, expected) in enumerate(CLIPS):
    video = videos[name]
    for col, index in enumerate(np.linspace(0, len(video) - 1, 5).astype(int)):
        axes[row, col].imshow(video[index])
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        axes[row, col].set_title(f"{index / SAMPLE_FPS:.1f}s", fontsize=8)
    axes[row, 0].set_ylabel(f"{name}\n({expected})", fontsize=8)
plt.tight_layout()

## Load the model

Loaded in bf16 and moved to the Mac GPU (`mps`). Like Marlin, Qwen3.5 falls back to reference implementations of two CUDA-only kernels on a Mac, which works but is slower.

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-2B"

start = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("mps").eval()
print(f"Loaded in {time.time() - start:.1f}s")

## The prompt

One generic prompt for every clip. The rules are the camera's configuration: the same prompt works for any camera, only the rule list changes. The model describes the clip first, then names the triggered rules, and returns an empty list when nothing matches.

In [ ]:
RULES = {
    "fighting": "people hit, kick, push or wrestle each other, or someone is physically assaulted",
    "robbery_or_theft": "someone takes a bag, phone, vehicle or other property that isn't theirs, from a person or a place, or threatens someone to take it",
    "break_in": "someone forces, pries or tampers with a door, window, shutter or lock to get into a closed place",
    "shoplifting": "a customer hides store merchandise in a bag or clothing instead of paying for it",
}

PROMPT = """You are the analyst for a fixed security camera. Watch the whole clip carefully.

Alert rules for this camera:
{rules}

First describe what happens in the clip in 2 to 4 sentences, in order. Then decide which alert rules, if any, were triggered. Most clips show ordinary activity: only report a rule if you clearly see it happen.

Reply with JSON only, in this format:
{{"description": "...", "triggered_rules": ["rule_name"]}}
Use an empty list if no rule was triggered.""".format(rules="\n".join(f"- {name}: {text}" for name, text in RULES.items()))

print(PROMPT)

In [ ]:
def analyze(video, thinking=False, max_new_tokens=2048):
    messages = [{"role": "user", "content": [{"type": "video"}, {"type": "text", "text": PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    metadata = VideoMetadata(
        total_num_frames=len(video),
        fps=SAMPLE_FPS,
        width=video.shape[2],
        height=video.shape[1],
        duration=len(video) / SAMPLE_FPS,
        frames_indices=list(range(len(video))),
    )
    inputs = processor(
        text=[text],
        videos=[video],
        video_metadata=[metadata],
        return_tensors="pt",
        videos_kwargs={"do_sample_frames": False, "do_resize": False},
    ).to(model.device)

    prompt_tokens = inputs["input_ids"].shape[1]
    torch.mps.synchronize()
    start = time.time()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.mps.synchronize()
    seconds = time.time() - start

    raw = processor.batch_decode(output[:, prompt_tokens:], skip_special_tokens=True)[0]
    answer = re.sub(r"(?s)^.*</think>", "", raw).strip()  # drop the reasoning, keep the final answer
    match = re.search(r"(?s)\{.*\}", answer)
    try:
        parsed = json.loads(match.group(0)) if match else None
    except json.JSONDecodeError:
        parsed = None
    return {
        "description": parsed.get("description") if parsed else answer,
        "triggered_rules": parsed.get("triggered_rules", []) if parsed else None,
        "valid_json": parsed is not None,
        "seconds": seconds,
        "input_tokens": prompt_tokens,
        "output_tokens": output.shape[1] - prompt_tokens,
        "raw": raw,
    }

## Run: thinking off

Greedy decoding (`do_sample=False`) so results are reproducible. For each clip we score two things:

- **alert correct**: it alerted on an event clip, or stayed quiet on a normal clip.
- **rule correct**: on event clips, the expected rule is among the triggered rules.

In [ ]:
def score_alert(expected, rules):
    if expected == "ambiguous":
        return None
    return bool(rules) == (expected is not None)


def score_rule(expected, rules):
    if expected == "ambiguous":
        return None
    return not rules if expected is None else expected in (rules or [])


def run(thinking):
    rows = []
    for name, _, _, expected in CLIPS:
        result = analyze(videos[name], thinking=thinking)
        rows.append({"clip": name, "expected": expected, "frames": len(videos[name]), **result})
        print(f"{name:20} {result['seconds']:5.1f}s  triggered={result['triggered_rules']}")
    df = pd.DataFrame(rows)
    expected = [None if pd.isna(e) else e for e in df["expected"]]  # pandas turns None into NaN
    df["alert_correct"] = [score_alert(e, rules) for e, rules in zip(expected, df["triggered_rules"])]
    df["rule_correct"] = [score_rule(e, rules) for e, rules in zip(expected, df["triggered_rules"])]
    return df


no_thinking = run(thinking=False)

In [ ]:
pd.set_option("display.max_colwidth", None)
no_thinking[["clip", "expected", "triggered_rules", "description"]]

## Run: thinking on

In [ ]:
thinking = run(thinking=True)

In [ ]:
thinking[["clip", "expected", "triggered_rules", "description"]]

## Run: more frames per second

At 2 fps a quick hand movement (prying a shutter, slipping an item into a bag) can fall between sampled frames. Here we sample at 4 fps, thinking off. It doubles the video tokens. (Upscaling the 320×240 frames to the full pixel budget made no difference in a separate test, so it isn't repeated here.)


In [ ]:
SAMPLE_FPS = 4.0
videos = {name: load_clip(name, window) for name, _, window, _ in CLIPS}
four_fps = run(thinking=False)
SAMPLE_FPS = 2.0


In [ ]:
four_fps[["clip", "expected", "triggered_rules", "description"]]


## Summary

In [ ]:
def summarize(df):
    scored = df[df["expected"] != "ambiguous"]
    events, normals = scored[scored["expected"].notna()], scored[scored["expected"].isna()]
    count = lambda series: int(series.astype(bool).sum())
    return {
        "events alerted": f"{count(events['alert_correct'])}/{len(events)}",
        "right rule on events": f"{count(events['rule_correct'])}/{len(events)}",
        "normals quiet": f"{count(normals['alert_correct'])}/{len(normals)}",
        "valid JSON": f"{int(df['valid_json'].sum())}/{len(df)}",
        "median seconds/clip": round(df["seconds"].median(), 1),
        "median output tokens": int(df["output_tokens"].median()),
    }


pd.DataFrame({"2 fps, thinking off": summarize(no_thinking), "2 fps, thinking on": summarize(thinking), "4 fps, thinking off": summarize(four_fps)})

In [ ]:
pd.DataFrame(
    {
        label: df.set_index("clip").loc["Arrest001", ["triggered_rules", "description"]]
        for label, df in [("2 fps, thinking off", no_thinking), ("2 fps, thinking on", thinking), ("4 fps, thinking off", four_fps)]
    }
)


## Takeaways

Results from our run (greedy decoding, so they're reproducible on the same machine). Ten clips is a small sample: this shows how the model behaves, not a reliable accuracy figure.

| | 2 fps | 2 fps + thinking | 4 fps |
| --- | --- | --- | --- |
| Normal clips left quiet | 4/4 | 4/4 | 4/4 |
| Event clips that raised an alert | 2/5 | 2/5 | 4/5 |
| Event clips with the right rule | 1/5 | 1/5 | 1/5 |
| Median time per clip (7–21 s clips) | 5.2 s | 22.8 s | 8.1 s |

**Describe:** good at the scene, the people and the broad action (who is where, what they wear, who walks where). Weak at small hand-object details in low-resolution CCTV: the shutter being forced became "interacts briefly with a yellow kiosk", and the shoplifter "bends down to pick up a paper bag".

**Understand complex situations:** it recognises violence well (the fight, and the scuffle in the arrest clip). It mostly misses intent: theft, tampering and shoplifting look like ordinary activity to it. At 4 fps it noticed more (the robber pulling out a weapon, someone opening the kiosk) but also hallucinated a weapon in the motorbike theft.

**Alert when something happens:** it stays quiet on normal footage, including the busy supermarket checkout, in every setting. At 4 fps it raised an alert on 4 of 5 events, but it rarely picks the right rule: physical events all become "fighting", and the break-in became "shoplifting". It's better at "something is wrong here" than at "this is what's wrong".

**Thinking mode:** same decisions, 4× slower. Not worth it here.

**Speed:** on a Mac, a 7–21 s clip takes about 5 s at 2 fps and 8 s at 4 fps, faster than real time, with ~2–4k input tokens per clip.

**For the pipeline:**
- Qwen3.5-2B works as a generic gate: one prompt with the camera's rules, an empty list when nothing happens. Treat any triggered rule as "alert-worthy" and don't trust *which* rule it names.
- Sample at 4 fps; 2 fps misses fast hand movements.
- These UCF-Crime clips are 320×240, a worst case. Modern cameras plus cropping to the tracked person (as in the Marlin spike) should help with the small details; the next test should use your own footage.
- If rule-level accuracy matters, try a larger Qwen3.5 model, or let Qwen only describe and have a separate decision step judge each rule.